In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError

import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher 

In [2]:
# Define NCBI error handling decorator with tenacity
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

#Create function to retrieve ALL pmids (with NCBI error handling)
@retry_on_communication_error
def Get_list(query):
    fetch = PubMedFetcher()  
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetch.pmids_for_query(query, 
                                      retstart=start_index,
                                      retmax=num_of_articles,
                                      pmc_only= False)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break     
    return(pmids)

In [3]:
#Read-in query version
with open("PUBMED_query_v1.2", "r") as f:
    file = []
    for line in f:
        file.append(line.replace('\t','').replace('\n','').strip())
query = " ".join(file[1:])

In [4]:
#Run query

a = datetime.now()
START = "2000-01-01"
STOP = "2024-04-01"
start_date_str = START
pmid_list = []
while True: #define periods so that <10,000 are retrieved in the least request calls (best speed)
    if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"): 
        month_interval = 6
    elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
        month_interval = 5
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01")):
        month_interval = 4
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01")):
        month_interval = 3
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01")):
        month_interval = 2
    else:
        month_interval = 4
    next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
    end_date = (next_start - relativedelta(days=1))
    end_date_str = end_date.strftime('%Y-%m-%d')
    date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
    pmids = Get_list(date_str+query)
    pmids_s = list(set(pmids))
    print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
    pmid_list.extend(pmids_s)
    start_date_str = next_start.strftime('%Y-%m-%d')
    if next_start>=date.fromisoformat(STOP):
        end_date_str = STOP
        break
pmids = Get_list(query)
pmids_s = list(set(pmids))
pmid_list.extend(pmids_s)
print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
print("Total query duration: ", datetime.now()-a)

2000-01-01 - 2000-06-30 :  7415 (7415)
2000-07-01 - 2000-12-31 :  7390 (7390)
2001-01-01 - 2001-06-30 :  8424 (8424)
2001-07-01 - 2001-12-31 :  7984 (7984)
2002-01-01 - 2002-06-30 :  8711 (8711)
2002-07-01 - 2002-12-31 :  8253 (8253)
2003-01-01 - 2003-05-31 :  7956 (7956)
2003-06-01 - 2003-10-31 :  7778 (7778)
2003-11-01 - 2004-03-31 :  8742 (8742)
2004-04-01 - 2004-08-31 :  8197 (8197)
2004-09-01 - 2005-01-31 :  9518 (9518)
2005-02-01 - 2005-06-30 :  8623 (8623)
2005-07-01 - 2005-11-30 :  9003 (9003)
2005-12-01 - 2006-03-31 :  8532 (8532)
2006-04-01 - 2006-07-31 :  7762 (7762)
2006-08-01 - 2006-11-30 :  7995 (7995)
2006-12-01 - 2007-03-31 :  9720 (9720)
2007-04-01 - 2007-07-31 :  8385 (8385)
2007-08-01 - 2007-11-30 :  8610 (8610)
2007-12-01 - 2008-03-31 :  9417 (9417)
2008-04-01 - 2008-07-31 :  8834 (8834)
2008-08-01 - 2008-11-30 :  8756 (8756)
2008-12-01 - 2009-03-31 :  9725 (9725)
2009-04-01 - 2009-07-31 :  8827 (8827)
2009-08-01 - 2009-11-30 :  8803 (8803)
2009-12-01 - 2010-02-28 :

In [5]:
#Issue

len(set(pmid_list))==len(pmid_list)

False

In [6]:
#Issue

#Confirm the same number is retrieved via Advanced Search at https://pubmed.ncbi.nlm.nih.gov/advanced/ 
len(set(pmid_list)) 
# ON THE SAME DAY March 29th, 2024, query up until now : 496,510
# NOT THE SAME

506509

In [7]:
#save for permanent storage after issues are resolved:
date_tag = datetime.now().isoformat()[:10]
#np.savetxt('PMID_lists/pmids_'+ date_tag +.txt', list(set(pmid_list)), delimiter=",")